# Query the consolidated data
Load `all_consolidated.csv`, drop the redundant whole-sample aggregates so only
the granular data remains, then select and show data SQL-style (by scale,
subscale, item, sample_type, subsample, data_type).

The query logic lives in `src/query.py` (readable functions); this notebook
just drives it. Run the cells top to bottom.

## 1. Setup: imports and load the data

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))   # so we can import query.py

import pandas as pd
from query import select, distinct, overview

# Show all columns when printing.
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
#df_all = pd.read_csv(ROOT / "data" / "all_consolidated_corr.csv", index_col=0)
#df_all = pd.read_csv(ROOT / "data" / "all_consolidated_corr.csv")
df_all = pd.read_csv(ROOT / "data" / "all_consolidated.csv")
print(f"loaded {len(df_all)} rows, {len(df_all.columns)} columns")

In [ ]:
from aggregate import aggregate

In [ ]:
def aggregate_over_subsamples(df, data_type="mean"):
    """Weighted mean across subsamples, giving one value per
    scale / subscale / record_type / sample_type.

    Pools a sample_type's subsamples together (drops `subsample` from the
    grouping), weighting each by its sample_size:
        weighted = sum(value * n) / sum(n)

    Assumes redundant aggregates have already been removed, so the subsamples
    don't overlap. Only `mean`/`median` should be passed (averaging SDs is
    not valid).
    """
    # keep only the statistic we want (e.g. the means)
    subset = df[df["data_type"] == data_type].copy()

    # make sure value and weight are numeric
    subset["value"] = pd.to_numeric(subset["value"], errors="coerce")
    subset["sample_size"] = pd.to_numeric(subset["sample_size"], errors="coerce")
    subset = subset.dropna(subset=["value", "sample_size"])

    # group WITHOUT subsample -> pools subsamples together
    group_cols = ["scale", "subscale", "record_type", "sample_type"]

    results = []
    for key, g in subset.groupby(group_cols):
        weighted = (g["value"] * g["sample_size"]).sum() / g["sample_size"].sum()
        row = dict(zip(group_cols, key))
        row["weighted_value"] = weighted
        row["n_groups"] = len(g)
        row["total_n"] = g["sample_size"].sum()
        results.append(row)

    return pd.DataFrame(results)

#### Any column name of the data frame works as an input value for the select function:

In [ ]:
help(select)

## 2. Drop the redundant aggregates
`redundant_aggregate == True` marks whole-sample values that overlap their own
subsamples. We remove them so aggregation/selection uses only the granular
(non-overlapping) data. Everything below works on `df`.

In [ ]:
# Keep only the rows that are NOT redundant aggregates.
df = df_all[df_all["redundant_aggregate"] == False]
print(f"kept {len(df)} of {len(df_all)} rows after dropping redundant aggregates")

## 3. Orientation: what's in the table?
Before querying, see how many distinct values each key column has, and list the
values you might filter on.

In [ ]:
overview(df)

In [ ]:
df.head(3)

In [ ]:
# List the distinct values of any column you want to filter on:
print("scales:     ", distinct(df, "scale"))
print("subscales:  ", distinct(df, "subscale"))
print("record types:", distinct(df, "record_type"))
print("sample types:", distinct(df, "sample_type"))
print("data types: ", distinct(df, "data_type"))

## 4. Example: DES-T

In [ ]:
df_filtered = select(df,
      scale='DES_T', 
      data_type='mean',
      record_type='total')
df_show = df_filtered[['publication', 
           'data_type', 
           'sample_type', 
           'subsample', 
           'sample_size', 
           'value', 
           'scale', 
           'scoring_rule']]
df_show

In [ ]:
df_filtered_modestin = df_filtered[df_filtered.publication=='Modestin & Erni 2004']
aggregate_over_subsamples(df_filtered_modestin, data_type="mean")


In [ ]:
aggregate_over_subsamples(df_filtered, data_type="mean")


## 5. Example CTQ-SF

In [ ]:
df_filtered = select(df,
      scale='CTQ_SF', 
      data_type='mean',
      record_type='total')
df_show = df_filtered[['publication', 
           'data_type', 
           'sample_type', 
           'subsample', 
           'sample_size', 
           'value', 
            'scale', 
            'subscale',
            'record_type',
           'scoring_rule']]
df_show

In [ ]:
aggregate_over_subsamples(df_filtered, data_type="mean")

## 6. Aggregating before or after data slicing:

In [ ]:
# Select DES-T means: 
des_t_means = select(df, scale="DES_T", data_type="mean")
set(des_t_means.sample_type)

In [ ]:
# Aggregate over subsamples:
aggregate_over_subsamples(des_t_means, data_type="mean")

In [ ]:
# Compare to aggregate() function:
grouped_df = aggregate(df, "mean", group_cols=["scale", "subscale", "record_type", "sample_type"])
grouped_df[grouped_df.scale=='DES_T']


### Aggregate DES-T means from one publication:

In [ ]:
# Compare slicing des_t_means data frame to directly selecting:
des_t_means_modestin = des_t_means[des_t_means.publication=='Modestin & Erni 2004']
print(des_t_means_modestin.shape)
select(df, scale="DES_T", data_type="mean", publication = 'Modestin & Erni 2004').shape

In [ ]:
des_t_means_modestin

In [ ]:
#select(df, scale="DES_T", data_type="mean", publication = 'Modestin & Erni 2004')

In [ ]:
# Aggregate over subsamples 
# (should be roughly equal to the redundant aggregate 
# from the publication as the scoring rule is an unweighted mean):
print(aggregate_over_subsamples(des_t_means_modestin, data_type="mean"))
red_aggr_modestin = df_all[(df_all.publication=='Modestin & Erni 2004') & (df_all.data_type=='mean') & (df_all.subsample=='whole_sample')]
red_aggr_modestin[['scale', 'subscale', 'record_type', 'sample_type', 'value', 'sample_size']]


#### Note: The aggregate() function automatically removes redundant aggregates whereas the aggregate_over_subsamples() function assumes they have been removed alread!

In [ ]:
df_modestin = df[df.publication=='Modestin & Erni 2004']
grouped_df = aggregate(df_modestin, "mean", group_cols=["scale", "subscale", "record_type", "sample_type"])
grouped_df

In [ ]:
df_modestin = df_all[df_all.publication=='Modestin & Erni 2004']
grouped_df = aggregate(df_modestin, "mean", group_cols=["scale", "subscale", "record_type", "sample_type"])
grouped_df

In [ ]:
set(df.source_file)

In [ ]:
df.columns

In [ ]:
set(df.publication)